In [2]:
import lightkurve as lk
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# This is TIC 28230919 — scored 0.9986 by ExoMiner++
# ExoMiner says period = 4.8878 days
# Let's SEE what ExoMiner saw

tic = "TIC 19028197"
period = 3.3366
t0 = 1687.2058

print(f"Fetching {tic} from NASA...")
lc = lk.search_lightcurve(tic, mission="TESS", author="SPOC", 
                           sector=14).download()
lc_clean = lc.flatten(window_length=401).remove_outliers(sigma=4)

# Four panels showing the full story
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Raw data — what TESS recorded
axes[0,0].scatter(lc.time.value, lc.flux.value, s=0.5, alpha=0.5, color='steelblue')
axes[0,0].set_title('1. What TESS recorded (raw)', fontsize=13)
axes[0,0].set_xlabel('Time (days)')
axes[0,0].set_ylabel('Brightness')

# Panel 2: Cleaned — trends removed
axes[0,1].scatter(lc_clean.time.value, lc_clean.flux.value, s=0.5, alpha=0.5, color='darkorange')
axes[0,1].set_title('2. After cleaning (stellar trends removed)', fontsize=13)
axes[0,1].set_xlabel('Time (days)')
axes[0,1].set_ylabel('Brightness')
axes[0,1].axhline(1.0, color='gray', linestyle='--', alpha=0.3)

# Panel 3: Phase-folded — all orbits stacked
folded = lc_clean.fold(period=period, epoch_time=t0)
binned = folded.bin(time_bin_size=0.008)
axes[1,0].scatter(folded.phase.value, folded.flux.value, s=0.5, alpha=0.15, color='gray')
axes[1,0].plot(binned.phase.value, binned.flux.value, 'o-', color='red', 
               markersize=3, linewidth=2)
axes[1,0].set_xlim(-0.15, 0.15)
axes[1,0].set_title('3. All orbits stacked (phase-folded) — THE PLANET', fontsize=13)
axes[1,0].set_xlabel('Phase (days from transit center)')
axes[1,0].set_ylabel('Brightness')
axes[1,0].axhline(1.0, color='blue', linestyle='--', alpha=0.3)

# Panel 4: Zoomed transit with annotations
axes[1,1].scatter(folded.phase.value, folded.flux.value, s=0.5, alpha=0.15, color='gray')
axes[1,1].plot(binned.phase.value, binned.flux.value, 'o-', color='red', 
               markersize=4, linewidth=2.5)
axes[1,1].set_xlim(-0.12, 0.12)

# Annotate what each part means
depth = (1.0 - np.nanmin(binned.flux.value))
axes[1,1].annotate('Planet starts crossing star', 
                    xy=(-0.05, 1.0), fontsize=10, color='green',
                    arrowprops=dict(arrowstyle='->', color='green'),
                    xytext=(-0.1, 1.003))
axes[1,1].annotate('Planet fully in front\n(deepest point)', 
                    xy=(0.0, 1.0-depth), fontsize=10, color='red',
                    arrowprops=dict(arrowstyle='->', color='red'),
                    xytext=(0.04, 1.0-depth-0.003))
axes[1,1].annotate('Planet exits other side', 
                    xy=(0.05, 1.0), fontsize=10, color='green',
                    arrowprops=dict(arrowstyle='->', color='green'),
                    xytext=(0.06, 1.003))
axes[1,1].axhline(1.0, color='blue', linestyle='--', alpha=0.3)
axes[1,1].set_title('4. Annotated — what each part of the dip means', fontsize=13)
axes[1,1].set_xlabel('Phase (days from transit center)')
axes[1,1].set_ylabel('Brightness')

plt.suptitle(f'{tic} — ExoMiner++ Score: 0.9986 — Period: {period} days', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print(f"  WHAT YOU JUST SAW")
print(f"{'='*60}")
print(f"  Star:           {tic}")
print(f"  ExoMiner score: 0.9986 (99.86% confident it's a planet)")
print(f"  Orbital period: {period} days ({period*24:.1f} hours)")
print(f"  Transit depth:  {depth*100:.3f}%")
print(f"  Planet radius:  ~5.1 Earth radii (sub-Saturn)")
print(f"")
print(f"  Panel 1: Raw telescope data — messy, noisy, hard to see anything")
print(f"  Panel 2: Cleaned — removed the star's own brightness changes")
print(f"  Panel 3: Stacked every orbit on top of each other — NOW you see it")
print(f"  Panel 4: Same thing but labeled — entry, deepest point, exit")
print(f"")
print(f"  This planet is roughly the size of Neptune.")
print(f"  It orbits its star in under 5 days.")
print(f"  ExoMiner++ is 99.86% sure this is real.")
print(f"{'='*60}")

Fetching TIC 19028197 from NASA...


No data found for target "TIC 19028197".


AttributeError: 'NoneType' object has no attribute 'flatten'